In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

from dotenv import load_dotenv

load_dotenv()

CF_OUTPUTS = Path(os.getenv("CF_OUTPUTS"))
print(CF_OUTPUTS)
print(CF_OUTPUTS.is_dir())

/home/dyretna/Dokument/Code/GitHub/nightingale_projects/cf_bench/cf_outputs
True


In [2]:
GEN_1_RANDOM = CF_OUTPUTS / "gen1_models_explainers_constraints" / "random_search_exp"
GEN_1_RANDOM.is_dir()

True

In [3]:
batch_data = GEN_1_RANDOM / "XGB_highthres_2026-04-17" / "annotated.csv"
batch_df = pd.read_csv(batch_data)

### set PD options

In [4]:
pd.set_option("display.max_rows", None)

In [5]:
# dtypes
int_columns = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "dosprt", "hltprhc", "Nchanged"]
float_columns = ["bmi", "gower_distance", "risk_before", "predicted_risk_after", "cf_gen_time_sec"]


# Convert and fill NaN with empty strings
batch_df[int_columns] = batch_df[int_columns].astype("Int64").astype("string").fillna("")
batch_df[float_columns] = batch_df[float_columns].round(4).astype("string").fillna("")
batch_df["valid"] = batch_df["valid"].astype("string").fillna("")

In [6]:
batch_df = batch_df.drop(columns=["hltprhc", "target_risk"])
batch_df["cf_id"] = batch_df["cf_id"].replace({"original": "xin"})

In [7]:
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,41.13,,,,0.026,
1,0,cf_1,,,5,5,,,,,,0.0813,2,False,0.026,0.0267
2,0,cf_2,,,6,,4,,,,,0.2,2,False,0.026,0.0278
3,0,cf_3,,,,6,,,21.1844,,,0.0946,2,False,0.026,0.0631
4,0,cf_4,,,1,,,,26.9487,,,0.1664,2,False,0.026,0.0449
5,0,cf_5,,7,,,3,,,,,0.1625,2,False,0.026,0.0243
6,0,cf_6,,,,,,1,21.8062,,,0.1662,2,False,0.026,0.0198
7,0,cf_7,7,,6,,,,,,,0.2,2,True,0.026,0.0016
8,0,cf_8,6,2,,,,,,,,0.1083,2,True,0.026,0.0088
9,0,cf_9,,,,2,,,,,,0.0625,1,True,0.026,0.0099


In [8]:
from cf_bench.utils import select_one_cf_per_query

# Select one CF per observation
# prefers valid CFs without violations, and lowest Gower
single_cf_df = select_one_cf_per_query(batch_df)
single_cf_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,41.13,,,,0.026,
9,0,cf_9,,,,2,,,,,,0.0625,1,True,0.026,0.0099
1,1,xin,3,4,1,2,3,0,22.3757,0,0.16,,,,0.2497,
10,1,cf_1,,,,,,,20.3772,,,0.0184,1,True,0.2497,0.0664
2,2,xin,5,3,1,1,4,0,22.694,7,0.16,,,,0.218,
11,2,cf_10,,,,,,,28.5787,,,0.037,1,True,0.218,0.0271
3,3,xin,3,3,6,6,2,0,24.3418,1,48.81,,,,0.0653,
12,3,cf_4,,1,,,,,,,,0.0625,1,True,0.0653,0.0042
4,4,xin,5,4,2,7,1,0,21.2585,3,42.86,,,,0.0811,
13,4,cf_5,,,,,,,22.2222,,,0.0077,1,True,0.0811,0.0162
